# Лабораторная работа 1
## Классификация CIFAR-10 с помощью полносвязных нейронных сетей

## 1. Постановка задачи

В этой работе я сравнил три полносвязные нейронные сети для классификации изображений из датасета CIFAR-10. В нём есть 10 классов: самолёты, автомобили, птицы, кошки, олени, собаки, лягушки, лошади, корабли и грузовики.

Цель работы — получить accuracy выше 50% на тестовой выборке без использования CNN. Разрешены только слои `Input`, `Dense`, `Dropout` и регуляризация весов в слоях `Dense`.

Каждое изображение имеет размер `32 × 32 × 3`. Перед обучением оно преобразуется в вектор длины `3072`. После такого преобразования теряется информация о взаимном расположении соседних пикселей. Поэтому полносвязной сети сложнее работать с изображениями, чем свёрточной.

## 2. Импорты и фиксация random seed

In [ ]:
import json
import time
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

RUN_TRAINING = False

cwd = Path.cwd().resolve()
if (cwd / "lw" / "lw1").exists():
    LAB_DIR = cwd / "lw" / "lw1"
elif cwd.name == "lw1" and cwd.parent.name == "lw":
    LAB_DIR = cwd
else:
    raise RuntimeError("Запустите ноутбук из корня проекта или из папки lw/lw1")

MODELS_DIR = LAB_DIR / "models"
OUTPUTS_DIR = LAB_DIR / "outputs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "best_cifar10_model.keras"
METRICS_PATH = OUTPUTS_DIR / "lab1_metrics.json"

print("Папка лабораторной работы:", LAB_DIR)
print("Файл лучшей модели:", MODEL_PATH)

## 3. Загрузка и подготовка данных

Сначала загружаю CIFAR-10, затем привожу значения пикселей к диапазону `[0, 1]`. Распрямление выполняется через NumPy, без специального слоя Keras:

`32 × 32 × 3 = 3072` признака для одного изображения.

После этого дополнительно стандартизирую признаки, отделяю 20% обучающей выборки для валидации и кодирую метки классов с помощью one-hot encoding.

In [ ]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.cifar10.load_data()

print("Исходные размеры:")
print("X_train_raw:", X_train_raw.shape)
print("y_train_raw:", y_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_test_raw:", y_test_raw.shape)

X_train_raw = X_train_raw.astype("float32") / 255.0
X_test_raw = X_test_raw.astype("float32") / 255.0

X_train_flat = X_train_raw.reshape(-1, 32 * 32 * 3)
X_test_flat = X_test_raw.reshape(-1, 32 * 32 * 3)

feature_mean = X_train_flat.mean(axis=0, keepdims=True)
feature_std = X_train_flat.std(axis=0, keepdims=True) + 1e-7
X_train_flat = (X_train_flat - feature_mean) / feature_std
X_test_flat = (X_test_flat - feature_mean) / feature_std

X_train, X_val, y_train_labels, y_val_labels = train_test_split(
    X_train_flat,
    y_train_raw.ravel(),
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_raw.ravel(),
)

y_train = keras.utils.to_categorical(y_train_labels, 10)
y_val = keras.utils.to_categorical(y_val_labels, 10)
y_test = keras.utils.to_categorical(y_test_raw.ravel(), 10)

print("\nРазмеры после подготовки:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test_flat:", X_test_flat.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

Данные подготовлены: изображения распрямлены в векторы длины 3072, признаки нормализованы и стандартизированы, а метки классов закодированы one-hot векторами. Разбиение на train и validation воспроизводится за счёт фиксированного `SEED = 42`.

## 4. Вспомогательные функции

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(epochs, history.history["loss"], label="Train loss")
    axes[0].plot(epochs, history.history["val_loss"], label="Validation loss")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("Эпоха")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, history.history["accuracy"], label="Train accuracy")
    axes[1].plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
    axes[1].set_title(f"{title}: accuracy")
    axes[1].set_xlabel("Эпоха")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test_labels):
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    return float(accuracy_score(y_test_labels, y_pred))


def train_and_time(model, epochs, batch_size, callbacks=None):
    start = time.time()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks or [],
        verbose=1,
    )
    return history, time.time() - start

## 5. Модель 1: маленькая сеть и недообучение

In [ ]:
underfit_model = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

underfit_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

if RUN_TRAINING:
    underfit_history, underfit_time = train_and_time(
        underfit_model, epochs=15, batch_size=256
    )
    plot_history(underfit_history, "Маленькая модель")
    underfit_test_acc = evaluate_model(underfit_model, X_test_flat, y_test_raw.ravel())
    print(f"Test accuracy маленькой модели: {underfit_test_acc:.4f}")
else:
    print("Обучение пропущено: используются сохранённые метрики.")

Маленькая модель показала невысокое качество: `train accuracy = 0.5501`, `validation accuracy = 0.4558`, `test accuracy = 0.4549`.

Один скрытый слой на 32 нейрона не может хорошо описать сложные зависимости в цветных изображениях CIFAR-10. Качество на train и validation остаётся сравнительно низким, поэтому это пример недообучения.

## 6. Модель 2: большая сеть и переобучение

In [ ]:
overfit_model = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(1024, activation="relu"),
    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

overfit_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

if RUN_TRAINING:
    overfit_history, overfit_time = train_and_time(
        overfit_model, epochs=25, batch_size=256
    )
    plot_history(overfit_history, "Большая модель без регуляризации")
    overfit_test_acc = evaluate_model(overfit_model, X_test_flat, y_test_raw.ravel())
    print(f"Test accuracy большой модели: {overfit_test_acc:.4f}")
else:
    print("Обучение пропущено: используются сохранённые метрики.")

Большая модель без регуляризации показала `train accuracy = 0.8689`, но её `validation accuracy = 0.4940`, а `test accuracy = 0.4874`.

По графику обучения видно, что переобучение начинается примерно с 4-й эпохи. После этого `train accuracy` продолжает расти, а `validation loss` ухудшается. Сеть запоминает обучающие примеры, но это почти не помогает на новых данных.

## 7. Модель 3: регуляризация и callbacks

Для третьей модели я использовал несколько способов борьбы с переобучением:

- `Dropout` случайно отключает часть нейронов во время обучения, поэтому сеть меньше запоминает конкретные примеры.
- L2-регуляризация штрафует слишком большие веса и делает модель устойчивее.
- `EarlyStopping` завершает обучение, когда качество на валидации перестаёт улучшаться.
- `ReduceLROnPlateau` уменьшает learning rate после выхода на плато.
- `ModelCheckpoint` сохраняет лучшую модель по `val_accuracy`, а не последнюю.

In [ ]:
regularized_model = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(2048, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.4),
    layers.Dense(1024, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.35),
    layers.Dense(512, activation="relu", kernel_regularizer=keras.regularizers.l2(5e-5)),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])

regularized_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

if RUN_TRAINING:
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=False, verbose=1
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH), monitor="val_accuracy", save_best_only=True, verbose=1
    )

    regularized_history, regularized_time = train_and_time(
        regularized_model,
        epochs=70,
        batch_size=256,
        callbacks=[early_stopping, reduce_lr, checkpoint],
    )
    plot_history(regularized_history, "Модель с регуляризацией")
else:
    print("Обучение пропущено: используется сохранённая лучшая модель.")

Обучение регуляризованной модели остановилось на 69-й эпохе. Лучший чекпоинт был сохранён при `val_accuracy = 0.5819`. Снижение learning rate после выхода на плато заметно помогло модели улучшить качество.

## 8. Оценка лучшей модели на тестовой выборке

In [ ]:
best_model = keras.models.load_model(MODEL_PATH)

y_pred_proba = best_model.predict(X_test_flat, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
test_accuracy = accuracy_score(y_test_raw.ravel(), y_pred)

print(f"Final test accuracy: {test_accuracy:.4f}")
print("Порог 50% преодолён:", "Да" if test_accuracy > 0.50 else "Нет")

Final test accuracy: 0.5682
Порог 50% преодолён: Да


## 9. Сравнение моделей

In [ ]:
with METRICS_PATH.open("r", encoding="utf-8") as file:
    saved_metrics = json.load(file)

results_df = pd.DataFrame(saved_metrics["results"])
display(results_df)

Модель,Описание,Эпох,Train accuracy,Validation accuracy,Test accuracy,"Время обучения, сек",Комментарий
Маленькая Dense-сеть,Input -> Dense(32) -> Dense(10),15,0.5501,0.4558,0.4549,15.5,Недообучение
Большая Dense-сеть без регуляризации,Input -> Dense(1024) -> Dense(512) -> Dense(256) -> Dense(10),25,0.8689,0.4940,0.4874,361.0,Переобучение примерно с 4-й эпохи
Регуляризованная Dense-сеть,"Dense(2048, 1024, 512, 256) + Dropout + L2 + callbacks",69,0.7794,0.5800,0.5682,2718.4,Лучшая модель


## 10. Итоговые выводы

В лабораторной работе я сравнил три полносвязные модели для классификации CIFAR-10.

Маленькая сеть с одним скрытым слоем на 32 нейрона оказалась слишком простой и недообучилась: её `test accuracy` равна `0.4549`. Большая сеть без регуляризации лучше подстроилась под train, но начала переобучаться примерно с 4-й эпохи. У неё `train accuracy` выросла до `0.8689`, а `test accuracy` осталась на уровне `0.4874`.

Лучший результат дала третья модель с Dropout, L2-регуляризацией и callbacks. После загрузки лучшего чекпоинта её `test accuracy` составила `0.5682`. Порог 50% преодолён.

Dense-сети не очень хорошо подходят для изображений: после преобразования картинки в плоский вектор они не используют локальную структуру пикселей. Обычно CNN решают эту задачу лучше, но в рамках ограничения лабораторной работы удалось получить результат выше 50%.